# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset title and description
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"License: {meta.license}")
print(f"Published: {meta.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets, their @id, and contained fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs.name if hasattr(rs, 'name') else ''} @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name if hasattr(field, 'name') else ''} (@id: {field.id}) | DataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record_set '@id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Available record set @ids: {record_set_ids}")
dataframes = {}

# Load records for each record set
for record_set_id in record_set_ids:
    print(f"\nLoading records for record_set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {record_set_id}.")

# Use the first available record set for further exploration
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nProceeding with record_set: {main_record_set_id}")
    print(f"Fields: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()
else:
    print("No tabular data found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field to analyze, e.g., 'cr:MW_kDa' (Molecular Weight in kDa)
# Please update the numeric_field_id and group_field_id as revealed by section 2 above.

record_set_id = main_record_set_id if 'main_record_set_id' in locals() else None
if record_set_id:
    df = dataframes[record_set_id]

    # Try to pick a numeric field (example, replace with actual field @id from above)
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
    else:
        # Try to infer numeric fields
        numeric_cols = []
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                numeric_cols.append(col)
            except:
                continue
        numeric_field = numeric_cols[0] if numeric_cols else df.columns[0]

    print(f"Using numeric field: {numeric_field}\n")

    # Filtering
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalizing the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field}:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Grouping by a categorical field, e.g., 'cr:Protein_Class' (replace as appropriate)
    candidate_group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == 'O']
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id:
    # Histogram of the numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by the group field, if available
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated loading, overview, basic filtering, normalization, grouping, and visualization of a mass spectrometry protein dataset using the Croissant metadata schema and `mlcroissant` libraries.

Explore the various fields and adapt filtering/grouping/visualization steps for your analysis questions.